# Python prediction API demo

## Load model
Here you just simply load the `Predictor` class and initialize without any arguments, it will automatically download and cache the weights *(this can be changed with arguments, but the defaults are recommended)*, load the model and prepare the necessary readers, preprocessors etc.

In [1]:
from predict import Predictor

model = Predictor()

/home/asger/Repositories/mini_trainer/mini_trainer/classifier.py:502: UserWarning: Model configuration option hidden overriden by value stored in config: 768 ==> True
  warnings.warn(
/home/asger/micromamba/envs/mini_trainer/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


## Usage
### Prediction
The `Predictor` can be used incredibly simply; simply call the model on a `list`/`tuple` of paths (`str`) or preloaded images (`torch.Tensor`/`np.ndarray`, or prebatched).

***Note:** most models expect preloaded images to be pre-resized to a specific input size, and provided in uint8 - handling of these cases may become more seamless in the future.*

In [ ]:
import numpy as np
import torch
from PIL import Image

images = [
    "/home/asger/Repositories/mini_trainer/examples/global_lepi/images/1730446/066d9c42ebd3c5b19f42de576ff82f83867f9b16.jpg",
    "/home/asger/Repositories/mini_trainer/examples/global_lepi/images/1956897/4ae9a90723346aeaa8ae76712d1762c8d2847632.jpg"
]

# With one image
pred1 = model(images[0])
print(pred1)

img0_pil = Image.open(images[0]).resize((512, 512)) # Currently the size is only handled automatically for string inputs

# NumPy
img0_np = np.transpose(np.asarray(img0_pil), (2, 1, 0)).copy() # Make sure you use C-H-W, not H-W-C
print(model(img0_np))
# PyTorch
img0_tensor = torch.from_numpy(img0_np).clone()
print(model(img0_tensor))

# With multiple images
pred2 = model(images)
print(pred2)

<HierarchicalPrediction(topk=1)>
	| 4525420-(72.5%) / 1830507-(73.2%) / 8874-(93.0%) |
<HierarchicalPrediction(topk=1)>
	| 4525420-(48.8%) / 1830507-(49.2%) / 8874-(84.0%) |
<HierarchicalPrediction(topk=1)>
	| 4525420-(48.8%) / 1830507-(49.2%) / 8874-(84.0%) |
<HierarchicalPrediction(topk=1)>
	| 4525420-(72.1%) / 1830507-(72.8%) / 8874-(93.0%) |
	| 1956897-(91.5%) / 1956895-(91.5%) / 6950-(99.8%) |


### Results
The returned object will always inherit from `mini_trainer.classifier.Prediction`, but may be a different subclass depending on the model (in particular `mini_trainer.hierarchical.model.HierarchicalPrediction`). 

#### Wrangling
This is extremely useful for quickly checking the results:

In [19]:
import os

import numpy as np

from mini_trainer.utils.io import is_image

dir = "/home/asger/Repositories/mini_trainer/examples/global_lepi/images/12258984"
images = [p for n in os.listdir(dir) if is_image(p := os.path.join(dir, n))]

pred = model(images)
print(str(pred)[:250] + "...")

correct = np.array(pred.labels)[:, 0, 0] == os.path.basename(dir)
print(f'Accuracy: {np.mean(correct).item():.1%} ({int(np.sum(correct).item())}/{len(images)})')

<HierarchicalPrediction(topk=1)>
	| 12258984-(97.3%) / 1924221-(99.5%) / 5473-(99.7%) |
	| 12258984-(99.6%) / 1924221-(99.7%) / 5473-(99.9%) |
	| 12258984-(96.6%) / 1924221-(99.2%) / 5473-(99.7%) |
	| 12258984-(88.4%) / 1924221-(95.7%) / 5473-(98.9%)...
Accuracy: 92.0% (449/488)


#### Serialization/Storage
Or serialization via dict-JSON:

In [77]:
import json
import tempfile

# Convert to dict
print("Dictionary serialization:")
print(*pred.to_dict()[:3], "", sep="\n")

# Or save to disk (powered directly by json.dumps and pred.to_dict)
with tempfile.NamedTemporaryFile(suffix=".json") as file:
    pred.save(file.name)
    with open(file.name) as f:
        print(
            "Contents of stored predictions:\n",
            "".join(f.readlines())[:250] + "..."
        )
    with open(file.name) as f:
        reconstructed = json.load(f)


def is_equal(a, b):
    if hasattr(a, "__iter__") and len(a) > 1 and hasattr(b, "__iter__") and len(b) > 1:
        return all(map(lambda xy : is_equal(*xy), zip(a, b)))
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        min_abs = max(1e-6, min(map(abs, [a, b])))
        diff = abs(a - b)
        rel_diff = diff / min_abs
        if rel_diff < 1e-2:
            return True
    return a == b


def check_equal(d1_d2 : tuple[dict, dict]):
    d1, d2 = d1_d2
    if d1.keys() != d2.keys():
        return False
    keys = list(d1.keys())
    return all([is_equal(d1[k], d2[k]) for k in keys])


good_reconstruction = all(list(map(check_equal, zip(pred.to_dict(), reconstructed["results"]))))
print(f'Reconstruction {"succeeded" if good_reconstruction else "failed"}!')

Dictionary serialization:
{'label': ('12258984', '1924221', '5473'), 'confidence': (0.9726881980895996, 0.9951934218406677, 0.9969378709793091), 'index': (3580, 1363, 46)}
{'label': ('12258984', '1924221', '5473'), 'confidence': (0.9960779547691345, 0.9966933727264404, 0.9994814991950989), 'index': (3580, 1363, 46)}
{'label': ('12258984', '1924221', '5473'), 'confidence': (0.9659198522567749, 0.9915325045585632, 0.9971330165863037), 'index': (3580, 1363, 46)}

Contents of stored predictions:
 {
  "results": [
    {
      "label": [
        "12258984",
        "1924221",
        "5473"
      ],
      "confidence": [
        0.9726881980895996,
        0.9951934218406677,
        0.9969378709793091
      ],
      "index": [
        3580,
  ...
Reconstruction succeeded!
